# Prediction analysis

This is to analyse the predicted ages.

In [30]:
import os
# Temporary workaround: allow duplicate OpenMP runtimes so the kernel can start.
# NOTE: This is unsafe and can mask real issues. See safer fixes below.
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

# Save directory for biological metrics plots
save_dir = 'biological_metrics_MCIandAD_quadratic_corr'
os.makedirs(save_dir, exist_ok=True)

# Load the predicted ages
results_df = pd.read_csv('../../compare_model_quadratic/QSM+MD+Thickness_subject_level_mean_raw_and_corrected_bag_on_age.csv')

# Add the MCI and AD subjects or not
MCIandAD_df = pd.read_csv('../../compare_model_quadratic/QSM+MD+Thickness_subject_level_mean_raw_and_quadratic_corrected_bag_on_age_MCIandAD.csv')
# Combine the two dataframes
results_df = pd.concat([results_df, MCIandAD_df], ignore_index=True)

# Load metadata
METADATA_PATH = "../../../AD_DECODE_data3.xlsx"
PCs_PATH = "../../../metadata_with_PCs_imputed.xlsx"
metadata_df = pd.read_excel(METADATA_PATH, sheet_name='AD_DECODE_data2')
PCs_df = pd.read_excel(PCs_PATH)

# Merge results with metadata by 'subject_id' in result_df and 'MRI_Exam' in metadata_df
# subject_id is like 02110
# MRI_Exam is like 2110
# Convert 'MRI_Exam' to match 'subject_id' format
metadata_df['subject_id'] = metadata_df['MRI_Exam'].apply(lambda x: f"{int(x):05d}" if pd.notnull(x) else np.nan)
PCs_df['subject_id'] = PCs_df['MRI_Exam'].apply(lambda x: f"{int(x):05d}" if pd.notnull(x) else np.nan)
# Ensure both subject_id columns are strings for merging
results_df['subject_id'] = results_df['subject_id'].apply(lambda x: f"{int(x):05d}")
# Merge the dataframes
PCs_cols = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15',
            'PC16', 'PC17', 'PC18', 'PC19', 'PC20', 'PC21', 'PC22', 'PC23', 'PC24', 'PC25', 'PC26', 'PC27', 'PC28', 'PC29', 'PC30',
            'blood_missing']
merged_df = pd.merge(results_df, metadata_df, on='subject_id', how='left')
merged_df = pd.merge(merged_df, PCs_df[['subject_id'] + PCs_cols], on='subject_id', how='left')

## cBAG vs. cognitive score


In [31]:
# Plot the cBAG vs column from 'MOCA_TOTAL' to 'Delayed_paraphrase'
import seaborn as sns

cognitive_start = merged_df.columns.get_loc('MOCA_TOTAL')
cognitive_end = merged_df.columns.get_loc('Delayed_paraphrase') + 1
cognitive_columns = merged_df.columns[cognitive_start:cognitive_end]

# Convert all cognitive columns to numeric, coercing errors to NaN
for col in cognitive_columns:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

R2 = []
p_values = []

for column in cognitive_columns:
    # Drop rows with NaN in either column for this analysis
    x = merged_df[column]
    y = merged_df['bag_corr']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    # Ignore if x or y has identical values (no variance)
    if x.nunique() <= 1 or y.nunique() <= 1:
        R2.append(np.nan)
        p_values.append(np.nan)
        continue
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add a regression line with R2 and p-value
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1:
        r_squared = np.corrcoef(x, y)[0, 1] ** 2
        from scipy.stats import linregress
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
    else:
        r_squared = np.nan
        p_value = np.nan
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'cBAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('cBAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'{save_dir}/cBAG_vs_{column}.png')
    plt.close()

# Print the top 10 cognitive columns with the highest R2 values
cognitive_r2_df = pd.DataFrame({'cognitive_column': cognitive_columns, 'R2': R2, 'p_value': p_values})
cognitive_r2_df = cognitive_r2_df.sort_values(by='R2', ascending=False)
# Save the R2 and p-values to a CSV file
cognitive_r2_df.to_csv(f'{save_dir}/cBAG_vs_cognition_r2_p_values.csv', index=False)
# Print the top 10 cognitive columns with highest R² values
top_10_cognitive = cognitive_r2_df.head(10)
print("Top 10 cognitive columns with highest R² values:")
print(top_10_cognitive)
# Print the top 10 cognitive columns with lowest p-values
top_10_cognitive_p = cognitive_r2_df.sort_values(by='p_value').head(10)
print("\nTop 10 cognitive columns with lowest p-values:")
print(top_10_cognitive_p)


C:\Users\22679\AppData\Local\Temp\ipykernel_18592\1005551410.py:41: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')


Top 10 cognitive columns with highest R² values:
        cognitive_column        R2   p_value
17     bckwds_max_length  0.053203  0.066698
30    Delayed_paraphrase  0.052721  0.067979
25                 ufov2  0.043192  0.099373
16  bckwds_total_correct  0.039124  0.117183
23        letter_fluency  0.038190  0.121734
22            fluency_4x  0.035807  0.134249
2      Delay_BensonTotal  0.033519  0.147602
26                 ufov3  0.026620  0.197683
15        fwd_max_length  0.022550  0.236259
14     fwd_total_correct  0.020085  0.263963

Top 10 cognitive columns with lowest p-values:
        cognitive_column        R2   p_value
17     bckwds_max_length  0.053203  0.066698
30    Delayed_paraphrase  0.052721  0.067979
25                 ufov2  0.043192  0.099373
16  bckwds_total_correct  0.039124  0.117183
23        letter_fluency  0.038190  0.121734
22            fluency_4x  0.035807  0.134249
2      Delay_BensonTotal  0.033519  0.147602
26                 ufov3  0.026620  0.197683
15 

In [32]:
# Compute FDR correction for p-values
from statsmodels.stats.multitest import multipletests
_, corrected_p_values, _, _ = multipletests(p_values, method='fdr_bh')
# Save the FDR corrected p-value results to a CSV file
# Sort from lowest to highest p-value
cognitive_r2_df['corrected_p_value'] = corrected_p_values
cognitive_r2_df = cognitive_r2_df.sort_values(by='corrected_p_value')
cognitive_r2_df.to_csv(f'{save_dir}/cBAG_vs_cognition_r2_FDR_corrected_p_values.csv', index=False)


## BAG vs. cognitive score

In [33]:
# Plot the BAG vs column from 'MOCA_TOTAL' to 'Delayed_paraphrase'
import seaborn as sns

cognitive_start = merged_df.columns.get_loc('MOCA_TOTAL')
cognitive_end = merged_df.columns.get_loc('Delayed_paraphrase') + 1
cognitive_columns = merged_df.columns[cognitive_start:cognitive_end]

# Convert all cognitive columns to numeric, coercing errors to NaN
for col in cognitive_columns:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

R2 = []
p_values = []

for column in cognitive_columns:
    # Drop rows with NaN in either column for this analysis
    x = merged_df[column]
    y = merged_df['bag_raw']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    # Ignore if x or y has identical values (no variance)
    if x.nunique() <= 1 or y.nunique() <= 1:
        R2.append(np.nan)
        p_values.append(np.nan)
        continue
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add a regression line with R2 and p-value
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1:
        r_squared = np.corrcoef(x, y)[0, 1] ** 2
        from scipy.stats import linregress
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
    else:
        r_squared = np.nan
        p_value = np.nan
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'BAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('BAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'{save_dir}/BAG_vs_{column}.png')
    plt.close()

# Print the top 10 cognitive columns with the highest R2 values
cognitive_r2_df = pd.DataFrame({'cognitive_column': cognitive_columns, 'R2': R2, 'p_value': p_values})
cognitive_r2_df = cognitive_r2_df.sort_values(by='R2', ascending=False)
cognitive_r2_df.to_csv(f'{save_dir}/BAG_vs_cognition_r2_p_values.csv', index=False)
top_10_cognitive = cognitive_r2_df.head(10)
print("Top 10 cognitive columns with highest R² values:")
print(top_10_cognitive)
# Print the top 10 cognitive columns with lowest p-values
top_10_cognitive_p = cognitive_r2_df.sort_values(by='p_value').head(10)
print("\nTop 10 cognitive columns with lowest p-values:")
print(top_10_cognitive_p)


C:\Users\22679\AppData\Local\Temp\ipykernel_18592\888801292.py:41: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')


Top 10 cognitive columns with highest R² values:
     cognitive_column        R2   p_value
23     letter_fluency  0.076778  0.026650
21        DigitSymbol  0.070456  0.034021
26              ufov3  0.055696  0.060462
25              ufov2  0.048272  0.081086
22         fluency_4x  0.046222  0.087998
12     RAVLT_LEARNING  0.039597  0.114944
8   Recognized_outof6  0.035623  0.135271
9         AVLT_Trial6  0.032592  0.153423
17  bckwds_max_length  0.031538  0.160350
11    RAVLT_IMMEDIATE  0.029339  0.175958

Top 10 cognitive columns with lowest p-values:
     cognitive_column        R2   p_value
23     letter_fluency  0.076778  0.026650
21        DigitSymbol  0.070456  0.034021
26              ufov3  0.055696  0.060462
25              ufov2  0.048272  0.081086
22         fluency_4x  0.046222  0.087998
12     RAVLT_LEARNING  0.039597  0.114944
8   Recognized_outof6  0.035623  0.135271
9         AVLT_Trial6  0.032592  0.153423
17  bckwds_max_length  0.031538  0.160350
11    RAVLT_IMMEDIATE

In [34]:
# Compute FDR correction for p-values
from statsmodels.stats.multitest import multipletests
_, corrected_p_values, _, _ = multipletests(p_values, method='fdr_bh')
# Save the FDR corrected p-value results to a CSV file
cognitive_r2_df['corrected_p_value'] = corrected_p_values
cognitive_r2_df = cognitive_r2_df.sort_values(by='corrected_p_value')
cognitive_r2_df.to_csv(f'{save_dir}/cBAG_vs_cognition_r2_FDR_corrected_p_values.csv', index=False)


## cBAG vs. biological metrics

In [35]:
import seaborn as sns
from scipy.stats import linregress

# Plot cBAG vs biological metrics with regression line, R2, and p-value

biological_start = merged_df.columns.get_loc('Systolic')
biological_end = merged_df.columns.get_loc('BMI') + 1
biological_columns = merged_df.columns[biological_start:biological_end]

R2 = []
p_values = []

for column in biological_columns:
    x = merged_df[column]
    y = merged_df['bag_corr']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add regression line
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1 and x.nunique() > 1 and y.nunique() > 1:
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        r_squared = r_value ** 2
    else:
        r_squared = float('nan')
        p_value = float('nan')
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'cBAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('cBAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'{save_dir}/cBAG_vs_{column}.png')
    plt.close()

biological_r2_df = pd.DataFrame({'biological_column': biological_columns, 'R2': R2, 'p_value': p_values})
biological_r2_df.to_csv(f'{save_dir}/cBAG_vs_biological_metrics_r2_p_values.csv', index=False)
# Compute FDR correction for p-values
from statsmodels.stats.multitest import multipletests
_, corrected_p_values, _, _ = multipletests(p_values, method='fdr_bh')
# Save the FDR corrected p-value results to a CSV file
# Sort from lowest to highest p-value
biological_r2_df['corrected_p_value'] = corrected_p_values
biological_r2_df = biological_r2_df.sort_values(by='corrected_p_value')
biological_r2_df.to_csv(f'{save_dir}/cBAG_vs_biological_metrics_FDR_corrected_p_values.csv', index=False)

C:\Users\22679\AppData\Local\Temp\ipykernel_18592\2337783272.py:32: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')


## BAG vs. biological metrics

In [36]:
import seaborn as sns
from scipy.stats import linregress

# Plot cBAG vs biological metrics with regression line, R2, and p-value

biological_start = merged_df.columns.get_loc('Systolic')
biological_end = merged_df.columns.get_loc('BMI') + 1
biological_columns = merged_df.columns[biological_start:biological_end]

R2 = []
p_values = []

for column in biological_columns:
    x = merged_df[column]
    y = merged_df['bag_raw']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add regression line
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1 and x.nunique() > 1 and y.nunique() > 1:
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        r_squared = r_value ** 2
    else:
        r_squared = float('nan')
        p_value = float('nan')
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'BAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('BAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'{save_dir}/BAG_vs_{column}.png')
    plt.close()

biological_r2_df = pd.DataFrame({'biological_column': biological_columns, 'R2': R2, 'p_value': p_values})
biological_r2_df.to_csv(f'{save_dir}/BAG_vs_biological_metrics_r2_p_values.csv', index=False)
# Compute FDR correction for p-values
from statsmodels.stats.multitest import multipletests
_, corrected_p_values, _, _ = multipletests(p_values, method='fdr_bh')
# Save the FDR corrected p-value results to a CSV file
# Sort from lowest to highest p-value
biological_r2_df['corrected_p_value'] = corrected_p_values
biological_r2_df = biological_r2_df.sort_values(by='corrected_p_value')
biological_r2_df.to_csv(f'{save_dir}/BAG_vs_biological_metrics_FDR_corrected_p_values.csv', index=False)

C:\Users\22679\AppData\Local\Temp\ipykernel_18592\1115379745.py:32: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')


## cBAG vs. PCs

In [37]:
# Plot the cBAG vs PC columns
import seaborn as sns

PC_columns = merged_df.columns[merged_df.columns.get_loc('PC1'):merged_df.columns.get_loc('PC30') + 1]

R2 = []
p_values = []

for column in PC_columns:
    # Drop rows with NaN in either column for this analysis
    x = merged_df[column]
    y = merged_df['bag_corr']
    mask = x.notna() & y.notna()
    # Ignore if 'blood_missing' is true
    mask = mask & (merged_df['blood_missing'] != 1)
    x = x[mask]
    y = y[mask]
    # Ignore if x or y has identical values (no variance)
    if x.nunique() <= 1 or y.nunique() <= 1:
        R2.append(np.nan)
        p_values.append(np.nan)
        continue
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add a regression line with R2 and p-value
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1:
        r_squared = np.corrcoef(x, y)[0, 1] ** 2
        from scipy.stats import linregress
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
    else:
        r_squared = np.nan
        p_value = np.nan
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'cBAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('cBAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'{save_dir}/cBAG_vs_{column}.png')
    plt.close()

# Print the top 10 PC columns with the highest R2 values
PC_r2_df = pd.DataFrame({'PC_column': PC_columns, 'R2': R2, 'p_value': p_values})
PC_r2_df = PC_r2_df.sort_values(by='R2', ascending=False)
# Save the R2 and p-values to a CSV file
PC_r2_df.to_csv(f'{save_dir}/cBAG_vs_PCs_r2_p_values.csv', index=False)
# Print the top 10 PC columns with highest R² values
top_10_PCs = PC_r2_df.head(10)
print("Top 10 PC columns with highest R² values:")
print(top_10_PCs)
# Print the top 10 PC columns with lowest p-values
top_10_PC_p = PC_r2_df.sort_values(by='p_value').head(10)
print("\nTop 10 PC columns with lowest p-values:")
print(top_10_PC_p)

C:\Users\22679\AppData\Local\Temp\ipykernel_18592\2164121165.py:37: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')


Top 10 PC columns with highest R² values:
   PC_column        R2   p_value
28      PC29  0.068844  0.093219
5        PC6  0.059960  0.118070
19      PC20  0.038856  0.210846
17      PC18  0.030551  0.268242
18      PC19  0.027986  0.289631
2        PC3  0.023017  0.337506
26      PC27  0.021656  0.352376
23      PC24  0.016198  0.421869
24      PC25  0.015580  0.430937
29      PC30  0.011096  0.506751

Top 10 PC columns with lowest p-values:
   PC_column        R2   p_value
28      PC29  0.068844  0.093219
5        PC6  0.059960  0.118070
19      PC20  0.038856  0.210846
17      PC18  0.030551  0.268242
18      PC19  0.027986  0.289631
2        PC3  0.023017  0.337506
26      PC27  0.021656  0.352376
23      PC24  0.016198  0.421869
24      PC25  0.015580  0.430937
29      PC30  0.011096  0.506751


## BAG vs. PCs

In [38]:
# Plot the cBAG vs PC columns
import seaborn as sns

PC_columns = merged_df.columns[merged_df.columns.get_loc('PC1'):merged_df.columns.get_loc('PC30') + 1]

R2 = []
p_values = []

for column in PC_columns:
    # Drop rows with NaN in either column for this analysis
    x = merged_df[column]
    y = merged_df['bag_raw']
    mask = x.notna() & y.notna()
    # Ignore if 'blood_missing' is true
    mask = mask & (merged_df['blood_missing'] != 1)
    x = x[mask]
    y = y[mask]
    # Ignore if x or y has identical values (no variance)
    if x.nunique() <= 1 or y.nunique() <= 1:
        R2.append(np.nan)
        p_values.append(np.nan)
        continue
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add a regression line with R2 and p-value
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1:
        r_squared = np.corrcoef(x, y)[0, 1] ** 2
        from scipy.stats import linregress
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
    else:
        r_squared = np.nan
        p_value = np.nan
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'BAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('BAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'{save_dir}/BAG_vs_{column}.png')
    plt.close()

# Print the top 10 PC columns with the highest R2 values
PC_r2_df = pd.DataFrame({'PC_column': PC_columns, 'R2': R2, 'p_value': p_values})
PC_r2_df = PC_r2_df.sort_values(by='R2', ascending=False)
# Save the R2 and p-values to a CSV file
PC_r2_df.to_csv(f'{save_dir}/BAG_vs_PCs_r2_p_values.csv', index=False)
# Print the top 10 PC columns with highest R² values
top_10_PCs = PC_r2_df.head(10)
print("Top 10 PC columns with highest R² values:")
print(top_10_PCs)
# Print the top 10 PC columns with lowest p-values
top_10_PC_p = PC_r2_df.sort_values(by='p_value').head(10)
print("\nTop 10 PC columns with lowest p-values:")
print(top_10_PC_p)

C:\Users\22679\AppData\Local\Temp\ipykernel_18592\643925440.py:37: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')


Top 10 PC columns with highest R² values:
   PC_column        R2   p_value
2        PC3  0.097075  0.044583
5        PC6  0.074389  0.080544
28      PC29  0.053782  0.139454
25      PC26  0.033298  0.247421
1        PC2  0.030958  0.265024
18      PC19  0.021398  0.355282
17      PC18  0.019150  0.382133
19      PC20  0.018805  0.386492
14      PC15  0.015798  0.427695
26      PC27  0.014653  0.445082

Top 10 PC columns with lowest p-values:
   PC_column        R2   p_value
2        PC3  0.097075  0.044583
5        PC6  0.074389  0.080544
28      PC29  0.053782  0.139454
25      PC26  0.033298  0.247421
1        PC2  0.030958  0.265024
18      PC19  0.021398  0.355282
17      PC18  0.019150  0.382133
19      PC20  0.018805  0.386492
14      PC15  0.015798  0.427695
26      PC27  0.014653  0.445082


## Volume analysis for cBAG

In [39]:
# Read the volume data
# BAG vs. Hippocampal Volume (Relative, z-scored)
# ===============================================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
from scipy.stats import zscore

# === Load raw regional volume data ===
vol_path = "../../normalized_regional_volumes.csv"
vol_df = pd.read_csv(vol_path)
# fill subject_id to 5 digits with leading zeros
vol_df['subject_id'] = vol_df['subject_id'].apply(lambda x: f"{int(x):05d}")
vol_df.head()

,subject_id,Left-Cerebellum-Cortex,Left-Thalamus-Proper,Left-Caudate,Left-Putamen,Left-Pallidum,Left-Hippocampus,Left-Amygdala,Left-Accumbens-area,Right-Cerebellum-Cortex,...,ctx-rh-rostralanteriorcingulate,ctx-rh-rostralmiddlefrontal,ctx-rh-superiorfrontal,ctx-rh-superiorparietal,ctx-rh-superiortemporal,ctx-rh-supramarginal,ctx-rh-frontalpole,ctx-rh-temporalpole,ctx-rh-transversetemporal,ctx-rh-insula
0,01912,0.034556,0.003690,0.002369,0.002884,0.000592,0.003499,0.001411,0.000316,0.036822,...,0.002127,0.012920,0.017552,0.007217,0.009087,0.008331,0.000601,0.001328,0.001008,0.006420
1,02110,0.033127,0.003674,0.002186,0.002398,0.000584,0.003233,0.001346,0.000278,0.034020,...,0.001679,0.013291,0.014513,0.008962,0.008608,0.008239,0.000862,0.001503,0.000760,0.004964
2,02224,0.038747,0.003222,0.001971,0.002499,0.000635,0.003526,0.001373,0.000232,0.038528,...,0.002370,0.010306,0.016758,0.008485,0.009313,0.009782,0.000671,0.001659,0.000734,0.005548
3,02227,0.029374,0.003562,0.001977,0.002867,0.000650,0.003325,0.001410,0.000268,0.033789,...,0.002650,0.010865,0.017221,0.007455,0.008871,0.007140,0.000613,0.001738,0.000740,0.005732
4,02373,0.026347,0.003688,0.003211,0.002315,0.000539,0.002857,0.001221,0.000301,0.026977,...,0.002648,0.013922,0.022962,0.007451,0.008105,0.009525,0.000841,0.001229,0.000951,0.006371


In [40]:
# Plot the BAG vs. volume for each region with regression line, R2, and p-value
# Prepare a helper to robustly format subject IDs as zero-padded 5-digit strings
def format_id(x):
    try:
        return f"{int(float(x)):05d}"
    except Exception:
        return np.nan

# Ensure subject_id in vol_df is preserved and formatted, and convert only volume columns to numeric
if 'subject_id' in vol_df.columns:
    vol_df['subject_id'] = vol_df['subject_id'].apply(format_id)

# Choose region columns (exclude subject_id)
region_columns = [c for c in vol_df.columns if c != 'subject_id']

# Convert region columns to numeric
for col in region_columns:
    vol_df[col] = pd.to_numeric(vol_df[col], errors='coerce')

R2 = []
p_values = []

# Ensure merged_df subject_id uses the same formatting
merged_df['subject_id'] = merged_df['subject_id'].apply(format_id)

# Merge on the now-consistent string subject_id
merged_vol_df = pd.merge(vol_df, merged_df[['subject_id', 'bag_corr']], on='subject_id', how='inner')

# Set 'subject_id' as index (optional)
merged_vol_df.set_index('subject_id', inplace=True)

for column in region_columns:
    x = merged_vol_df[column]
    y = merged_vol_df['bag_corr']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add regression line
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1 and x.nunique() > 1 and y.nunique() > 1:
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        r_squared = r_value ** 2
    else:
        r_squared = float('nan')
        p_value = float('nan')
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'cBAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('cBAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'{save_dir}/cBAG_vs_{column}.png')
    plt.close()

# Build the results DataFrame using the same region_columns (lengths match)
volume_r2_df = pd.DataFrame({'volume_column': region_columns, 'R2': R2, 'p_value': p_values})
volume_r2_df = volume_r2_df.sort_values(by='R2', ascending=False)
top_10_volume = volume_r2_df.head(10)
print("Top 10 volume columns with highest R² values:")
print(top_10_volume)
# Print the top 10 volume columns with lowest p-values
top_10_volume_p = volume_r2_df.sort_values(by='p_value').head(10)
print("\nTop 10 volume columns with lowest p-values:")
print(top_10_volume_p)

# Compute FDR correction for p-values robustly (handle NaNs)
from statsmodels.stats.multitest import multipletests
p_arr = np.array(p_values, dtype=float)
mask = np.isfinite(p_arr)
corrected = np.full_like(p_arr, np.nan, dtype=float)
if mask.any():
    corrected_p = multipletests(p_arr[mask], method='fdr_bh')[1]
    corrected[mask] = corrected_p

# Sort by p-value and save
volume_r2_df = volume_r2_df.sort_values(by='p_value').reset_index(drop=True)
volume_r2_df['corrected_p_value'] = corrected
volume_r2_df.to_csv(f'{save_dir}/volume_vs_cBAG_results.csv', index=False)


C:\Users\22679\AppData\Local\Temp\ipykernel_18592\435366885.py:51: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')


Top 10 volume columns with highest R² values:
                 volume_column        R2   p_value
36          ctx-lh-postcentral  0.097390  0.012058
24     ctx-lh-isthmuscingulate  0.075803  0.027671
73            ctx-rh-precuneus  0.051344  0.071782
39            ctx-lh-precuneus  0.043573  0.097862
43     ctx-lh-superiorparietal  0.042385  0.102663
76      ctx-rh-superiorfrontal  0.038698  0.119236
49               ctx-lh-insula  0.038491  0.120248
18  ctx-lh-caudalmiddlefrontal  0.036710  0.129352
56     ctx-rh-inferiorparietal  0.034570  0.141297
68     ctx-rh-parstriangularis  0.031630  0.159731

Top 10 volume columns with lowest p-values:
                 volume_column        R2   p_value
36          ctx-lh-postcentral  0.097390  0.012058
24     ctx-lh-isthmuscingulate  0.075803  0.027671
73            ctx-rh-precuneus  0.051344  0.071782
39            ctx-lh-precuneus  0.043573  0.097862
43     ctx-lh-superiorparietal  0.042385  0.102663
76      ctx-rh-superiorfrontal  0.038698  

## Volume analysis for BAG

In [41]:
# Plot the BAG vs. volume for each region with regression line, R2, and p-value
# Prepare a helper to robustly format subject IDs as zero-padded 5-digit strings
def format_id(x):
    try:
        return f"{int(float(x)):05d}"
    except Exception:
        return np.nan

# Ensure subject_id in vol_df is preserved and formatted, and convert only volume columns to numeric
if 'subject_id' in vol_df.columns:
    vol_df['subject_id'] = vol_df['subject_id'].apply(format_id)

# Choose region columns (exclude subject_id)
region_columns = [c for c in vol_df.columns if c != 'subject_id']

# Convert region columns to numeric
for col in region_columns:
    vol_df[col] = pd.to_numeric(vol_df[col], errors='coerce')

R2 = []
p_values = []

# Ensure merged_df subject_id uses the same formatting
merged_df['subject_id'] = merged_df['subject_id'].apply(format_id)

# Merge on the now-consistent string subject_id
merged_vol_df = pd.merge(vol_df, merged_df[['subject_id', 'bag_raw']], on='subject_id', how='inner')

# Set 'subject_id' as index (optional)
merged_vol_df.set_index('subject_id', inplace=True)

for column in region_columns:
    x = merged_vol_df[column]
    y = merged_vol_df['bag_raw']
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    plt.figure(figsize=(10, 6))
    plt.scatter(x, y, alpha=0.5)
    # Add regression line
    sns.regplot(x=x, y=y, scatter=False, color='red', line_kws={'label': 'Regression Line'})
    if len(x) > 1 and len(y) > 1 and x.nunique() > 1 and y.nunique() > 1:
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        r_squared = r_value ** 2
    else:
        r_squared = float('nan')
        p_value = float('nan')
    R2.append(r_squared)
    p_values.append(p_value)
    plt.title(f'BAG vs {column}')
    plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')
    plt.ylabel('BAG')
    plt.xlabel(column)
    plt.grid()
    plt.savefig(f'{save_dir}/BAG_vs_{column}.png')
    plt.close()

# Build the results DataFrame using the same region_columns (lengths match)
volume_r2_df = pd.DataFrame({'volume_column': region_columns, 'R2': R2, 'p_value': p_values})
volume_r2_df = volume_r2_df.sort_values(by='R2', ascending=False)
top_10_volume = volume_r2_df.head(10)
print("Top 10 volume columns with highest R² values:")
print(top_10_volume)
# Print the top 10 volume columns with lowest p-values
top_10_volume_p = volume_r2_df.sort_values(by='p_value').head(10)
print("\nTop 10 volume columns with lowest p-values:")
print(top_10_volume_p)

# Compute FDR correction for p-values robustly (handle NaNs)
from statsmodels.stats.multitest import multipletests
p_arr = np.array(p_values, dtype=float)
mask = np.isfinite(p_arr)
corrected = np.full_like(p_arr, np.nan, dtype=float)
if mask.any():
    corrected_p = multipletests(p_arr[mask], method='fdr_bh')[1]
    corrected[mask] = corrected_p

# Sort by p-value and save
volume_r2_df = volume_r2_df.sort_values(by='p_value').reset_index(drop=True)
volume_r2_df['corrected_p_value'] = corrected
volume_r2_df.to_csv(f'{save_dir}/volume_vs_BAG_results.csv', index=False)


C:\Users\22679\AppData\Local\Temp\ipykernel_18592\432810978.py:51: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(title=f'R² = {r_squared:.2f}, p = {p_value:.3f}')


Top 10 volume columns with highest R² values:
                 volume_column        R2   p_value
36          ctx-lh-postcentral  0.114930  0.006138
23     ctx-lh-inferiortemporal  0.074356  0.029261
77     ctx-rh-superiorparietal  0.070491  0.033976
9        Right-Thalamus-Proper  0.064239  0.043300
1         Left-Thalamus-Proper  0.062278  0.046736
62  ctx-rh-medialorbitofrontal  0.055111  0.061869
56     ctx-rh-inferiorparietal  0.055072  0.061964
24     ctx-lh-isthmuscingulate  0.054470  0.063449
57     ctx-rh-inferiortemporal  0.053501  0.065919
79        ctx-rh-supramarginal  0.053454  0.066041

Top 10 volume columns with lowest p-values:
                 volume_column        R2   p_value
36          ctx-lh-postcentral  0.114930  0.006138
23     ctx-lh-inferiortemporal  0.074356  0.029261
77     ctx-rh-superiorparietal  0.070491  0.033976
9        Right-Thalamus-Proper  0.064239  0.043300
1         Left-Thalamus-Proper  0.062278  0.046736
62  ctx-rh-medialorbitofrontal  0.055111  